[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/casos-de-uso-avanzados/03-structured-output.ipynb)

# Salidas Estructuradas con Instructor y Pydantic

> Aprende a extraer datos estructurados y validados de texto libre usando Instructor
> con Claude y modelos Pydantic, garantizando tipado y validación automática.

## Objetivos
- Definir modelos Pydantic para estructurar respuestas de LLMs
- Usar Instructor para extraer datos de facturas y emails
- Validar automáticamente la salida con Pydantic
- Manejar modelos anidados y listas de objetos

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic instructor pydantic --quiet

## 2. Configuración con Instructor

In [ ]:
import anthropic
import instructor
from pydantic import BaseModel, Field, validator
from typing import Optional
from datetime import date
from enum import Enum

# Instructor envuelve el cliente de Anthropic para devolver objetos Pydantic
cliente_raw = anthropic.Anthropic()
cliente = instructor.from_anthropic(cliente_raw)
MODELO = "claude-haiku-4-5-20251001"

print("Cliente Instructor inicializado.")
print("Instructor versión:", instructor.__version__)

## 3. Extraer datos de facturas

Definimos un modelo Pydantic que representa una factura y extraemos los datos automáticamente.

In [ ]:
class LineaFactura(BaseModel):
    descripcion: str = Field(description="Descripción del producto o servicio")
    cantidad: int = Field(description="Número de unidades", ge=1)
    precio_unitario: float = Field(description="Precio por unidad en euros", ge=0)
    total: float = Field(description="Total de la línea (cantidad × precio)")

class Factura(BaseModel):
    numero_factura: str = Field(description="Número o código de la factura")
    fecha: str = Field(description="Fecha de emisión en formato YYYY-MM-DD")
    proveedor: str = Field(description="Nombre o razón social del proveedor")
    cliente: str = Field(description="Nombre o razón social del cliente")
    lineas: list[LineaFactura] = Field(description="Lista de líneas de la factura")
    subtotal: float = Field(description="Subtotal sin IVA")
    iva_porcentaje: float = Field(description="Porcentaje de IVA aplicado")
    total_con_iva: float = Field(description="Total final con IVA incluido")

# Texto de factura en formato libre (como llegaría por email o OCR)
TEXTO_FACTURA = """
FACTURA Nº F-2024-0892
Fecha: 15 de marzo de 2024

Proveedor: Servicios Cloud SL — NIF B87654321
Cliente: Tech Startup SA — NIF A12345678

CONCEPTOS:
- Licencias software x 5 unidades a 299€/u = 1.495,00€
- Soporte técnico premium x 1 mes a 450€ = 450,00€
- Formación online x 3 sesiones a 150€ = 450,00€

Subtotal: 2.395,00€
IVA 21%: 502,95€
TOTAL: 2.897,95€
"""

factura = cliente.messages.create(
    model=MODELO,
    max_tokens=1024,
    messages=[{"role": "user", "content": f"Extrae los datos de esta factura:\n{TEXTO_FACTURA}"}],
    response_model=Factura
)

print("=== FACTURA EXTRAÍDA ===")
print(f"Número: {factura.numero_factura}")
print(f"Fecha: {factura.fecha}")
print(f"Proveedor: {factura.proveedor}")
print(f"Cliente: {factura.cliente}")
print(f"\nLíneas ({len(factura.lineas)}):")
for linea in factura.lineas:
    print(f"  - {linea.descripcion}: {linea.cantidad} × {linea.precio_unitario}€ = {linea.total}€")
print(f"\nSubtotal: {factura.subtotal}€")
print(f"IVA {factura.iva_porcentaje}%: {factura.total_con_iva - factura.subtotal:.2f}€")
print(f"TOTAL: {factura.total_con_iva}€")

## 4. Clasificar emails automáticamente

In [ ]:
class Prioridad(str, Enum):
    ALTA = "alta"
    MEDIA = "media"
    BAJA = "baja"

class CategoriaEmail(str, Enum):
    SOPORTE = "soporte"
    VENTAS = "ventas"
    RRHH = "rrhh"
    LEGAL = "legal"
    OTRO = "otro"

class EmailClasificado(BaseModel):
    categoria: CategoriaEmail
    prioridad: Prioridad
    accion_requerida: str = Field(description="Acción concreta que debe tomarse")
    destinatario_sugerido: str = Field(description="Departamento o persona que debe gestionar este email")
    resumen: str = Field(description="Resumen del email en una frase", max_length=150)
    requiere_respuesta_urgente: bool

emails = [
    "Estimados señores, llevamos 3 días sin poder acceder a nuestra cuenta. Tenemos una demo con un cliente importante mañana y necesitamos solucionar esto hoy. Por favor, contacten urgentemente.",
    "Hola, me gustaría recibir información sobre los planes Enterprise. Estamos evaluando opciones para nuestra empresa de 200 empleados. Sin prisa, tenemos 3 meses para decidir.",
    "Os envío el contrato revisado por nuestro departamento legal. Hay 2 cláusulas (4.2 y 7.1) que requieren revisión antes de firmar. Adjunto el documento."
]

print("=== CLASIFICACIÓN DE EMAILS ===")
for i, email in enumerate(emails, 1):
    resultado = cliente.messages.create(
        model=MODELO,
        max_tokens=512,
        messages=[{"role": "user", "content": f"Clasifica este email:\n{email}"}],
        response_model=EmailClasificado
    )
    print(f"\nEmail #{i}: '{email[:60]}...'")
    print(f"  Categoría: {resultado.categoria.value} | Prioridad: {resultado.prioridad.value}")
    print(f"  Acción: {resultado.accion_requerida}")
    print(f"  Para: {resultado.destinatario_sugerido}")
    print(f"  Urgente: {'✓ SÍ' if resultado.requiere_respuesta_urgente else '✗ No'}")

## 5. Ventajas de Instructor vs JSON manual

In [ ]:
# Instructor reintenta automáticamente si Pydantic lanza ValidationError
# Esto garantiza que siempre obtenemos un objeto válido

class ProductoExtracto(BaseModel):
    nombre: str
    precio: float = Field(ge=0, description="Precio en euros, nunca negativo")
    disponible: bool
    categorias: list[str] = Field(min_length=1, description="Al menos una categoría")

texto_producto = """
Auriculares inalámbricos Sony WH-1000XM5 con cancelación de ruido activa.
Precio de venta: 349 euros. Disponible en stock.
Categorías: electrónica, audio, premium, trabajo-desde-casa.
"""

producto = cliente.messages.create(
    model=MODELO,
    max_tokens=256,
    messages=[{"role": "user", "content": f"Extrae los datos de este producto:\n{texto_producto}"}],
    response_model=ProductoExtracto
)

print("=== PRODUCTO EXTRAÍDO Y VALIDADO ===")
print(f"Nombre: {producto.nombre}")
print(f"Precio: {producto.precio}€")
print(f"Disponible: {'Sí' if producto.disponible else 'No'}")
print(f"Categorías: {', '.join(producto.categorias)}")
print(f"\nTipo Python: {type(producto).__name__}")
print(f"Modelo Pydantic válido: ✓")
print(f"JSON exportable: {producto.model_dump_json(indent=2)}")